# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook explores the FAIR² dataset on clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors, using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the Croissant schema metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1mDataset Name: {metadata.name}\033[0m")
print(f"\033[1mDescription:\033[0m {metadata.description}")

## 2. Data Overview

Let us display the available record sets, their `@id`, and list the fields (`cr:field`) for each. All identifiers below are the actual `@id` as specified in the dataset schema.


In [ ]:
# List all record sets and their @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rset in record_sets:
        print(f"\nRecord Set: {rset.name}")
        print(f"  @id: {rset.id}")

        # List the fields (columns)
        if hasattr(rset, 'fields') and rset.fields:
            print("  Fields (@id):")
            for field in rset.fields:
                print(f"    - {field.name}: {field.id}")
        else:
            print("  No fields found.")

By examining the record sets above, you can identify which `@id` to use for loading records from a specific table. Below we provide a preview of the first few records for the *main* tabular record set in the FAIR² dataset, using its `@id`.

In [ ]:
# For this dataset, there is one main record set for the tabular clinical data:
# We extract the first available record set:
main_record_set = record_sets[0]
main_record_set_id = main_record_set.id

print(f"Loading records from main record set '{main_record_set.name}' (@id: {main_record_set_id})")

# Preview the records by @id
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction

Load all records from each discovered record set into one or more pandas DataFrames, using each record set's `@id`. This enables downstream analysis and manipulation using pandas.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Preview columns from the main record set
print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())

# Preview the first 5 records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Typical EDA steps: filtering records by threshold, normalizing numeric variables, grouping results.

**Note:** All columns/fields are always referenced by their full Croissant `@id` (not short names).

Let's identify a numeric field (e.g., patient age at 2nd diagnosis) and a key grouping field (e.g., sex/gender or MSI status).

In [ ]:
# List all columns and their names for reference
df = dataframes[main_record_set_id]
print("All columns in main record set:")
for c in df.columns:
    print(c)

# Select a likely numeric field by @id (example: 'Age_at_Second_CRC_Diagnosis' -- update as needed)
# Assume a field such as 'https://api.app.sen.science/frontiers/7862866/field/age_2nd_crc' is present.
# We'll use the first numeric column (float/integer dtype) discovered, for demonstration.
import numpy as np
numeric_field_id = None
for col in df.columns:
    # Check dtype or try conversion
    series = df[col]
    try:
        # Try float conversion for the entire column
        series_float = pd.to_numeric(series, errors='coerce')
        if series_float.notna().sum() > 2:  # at least a few non-null
            numeric_field_id = col
            break
    except Exception:
        continue

if numeric_field_id is None:
    raise ValueError("No numeric column found in this record set!")
print(f"Selected numeric field for EDA: {numeric_field_id}")

# Apply a simple threshold for demonstration (e.g., age > 60)
threshold = 60
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization
mean_ = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
std_ = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - mean_) / std_
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a likely categorical field (e.g., sex, anatomical_site, or MSI status)
# We'll choose the first object or category dtype column with <=10 unique values (excluding the numeric field)
group_field_id = None
for col in df.columns:
    if col == numeric_field_id:
        continue
    # Count unique non-null values
    nunique = df[col].dropna().nunique()
    if nunique > 1 and nunique <= 10:
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id, observed=True)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df)
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization

We visualize the distribution of the chosen numeric variable (e.g., age at second cancer diagnosis), as well as any key group comparison (e.g., MSI status or anatomical site), all referenced by their full Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field (for the full dataset and filtered)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=15, kde=True, ax=ax[0], color='skyblue')
ax[0].set_title(f'Distribution of {numeric_field_id}')
ax[0].set_xlabel(numeric_field_id)

sns.histplot(pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').dropna(), bins=10, kde=True, ax=ax[1], color='salmon')
ax[1].set_title(f'Filtered {numeric_field_id} (> {threshold})')
ax[1].set_xlabel(numeric_field_id)
plt.tight_layout()
plt.show()

# If group_field_id found, plot the group means
if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df, ci=None)
    plt.title(f'Mean {numeric_field_id} grouped by {group_field_id}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.show()

## 6. Conclusion

In this notebook, we have:

* Loaded the FAIR² clinical dataset using its Croissant schema and `mlcroissant`
* Explored available record sets and columns by their `@id`
* Loaded all record data into pandas DataFrames indexed by record set `@id`
* Performed basic EDA: filtering, normalization, grouping by key fields—always referencing data by their unique Croissant `@id`
* Visualized distributions and group comparisons from the clinical data

This approach ensures robust, interoperable data workflows leveraging Croissant standard and reproducible data referencing.